In [1]:
# Installs necessary libraries from Hugging Face and PyTorch
!pip install transformers datasets torch --quiet


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mlflow 2.10.0 requires pyarrow<16,>=4.0.0, but you have pyarrow 21.0.0 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [1]:
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from PIL import Image
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    CLIPModel,         # Added for multimodal section later
    CLIPProcessor      # Added for multimodal section later
)
from datasets import load_dataset
import requests        # Added for multimodal section later
from io import BytesIO # Added for multimodal section later


# Load dataset and model
dataset = load_dataset("imdb")  # Standard benchmark dataset for sentiment analysis
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2) # num_labels=2 for positive/negative sentiment


/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-10-22 16:19:12.573238: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-10-22 16:19:12.573291: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-10-22 16:19:12.575295: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-10-22 16:19:12.587407: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to 

In [2]:
# Speed optimization 1: Freeze embedding layers
for param in model.distilbert.embeddings.parameters():
    param.requires_grad = False


In [3]:
def tokenize_function(examples):
    # Tokenize text, pad shorter sequences to max_length, and truncate longer ones.
    # max_length=128 is chosen for speed; longer sequences might capture more context but take more memory/time.
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128) # Speed optimization 2: Limit max length

# Apply the tokenization function to the entire dataset in batches for efficiency.
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Select smaller subsets for faster training and evaluation during the demo.
# Shuffling ensures the subset is representative.
train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(500))  # Speed optimization 3: Even smaller dataset
eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(50))  # Reduced evaluation set (used later)


Map: 100%|███████████████████████| 25000/25000 [00:13<00:00, 1860.18 examples/s]


In [4]:
# Configuration for optimized training using the Trainer API
training_args = TrainingArguments(
    output_dir="./results",  # Directory for checkpoints and logs

    # Training strategy settings
    eval_strategy="no",  # Disable evaluation during training for speed (use "epoch" or "steps" for evaluation)
    num_train_epochs=1,  # Reduced epochs for faster demo

    # Batch size and gradient accumulation
    per_device_train_batch_size=16, # Batch size per GPU/CPU
    gradient_accumulation_steps=2,  # Update weights every 2 batches (effective batch size = 16*2=32)

    # Optimization settings
    fp16=True if torch.cuda.is_available() else False,  # Use mixed precision (faster, less memory) if CUDA is available

    # Logging and reporting
    report_to="none",  # Disable external logging (like Weights & Biases)
    logging_steps=10,  # Log training loss every 10 steps
)

# Create trainer instance using the prepared model and tokenized dataset
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
    # eval_dataset is omitted because eval_strategy="no"
)

# Start the fine-tuning process
trainer.train()


/usr/local/lib/python3.11/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,0.681600


TrainOutput(global_step=16, training_loss=0.666598379611969, metrics={'train_runtime': 171.7699, 'train_samples_per_second': 2.911, 'train_steps_per_second': 0.093, 'total_flos': 16558424832000.0, 'train_loss': 0.666598379611969, 'epoch': 1.0})

In [6]:
# Test the model with a sample input
inputs = tokenizer("This movie was absolutely fantastic!", return_tensors="pt")

# Get model predictions (logits)
# Note: The Trainer automatically handles device placement during training.
# For inference after Trainer, ensure inputs are on the same device as the model.
model.to("cpu") # Move model back to CPU if needed, or keep on GPU if available
inputs = {k: v.to(model.device) for k, v in inputs.items()} # Move inputs to model's device

outputs = model(**inputs)
print("Logits for 'fantastic':", outputs.logits) # Higher value for the positive class (index 1 probably) expected

# Test with a negative example
inputs_neg = tokenizer("This movie was terrible.", return_tensors="pt")
inputs_neg = {k: v.to(model.device) for k, v in inputs_neg.items()} # Move to model's device
outputs_neg = model(**inputs_neg)
print("Logits for 'terrible':", outputs_neg.logits) # Higher value for the negative class (index 0 probably) expected


Logits for 'fantastic': tensor([[-0.0602, -0.0546]], grad_fn=<AddmmBackward0>)
Logits for 'terrible': tensor([[0.0124, 0.0009]], grad_fn=<AddmmBackward0>)


In [7]:
# Re-define an optimizer for this specific manual loop demonstration
# (Trainer manages its own optimizer internally)
# Ensure model parameters requiring gradients are included (embeddings might still be frozen from Step 3)
trainable_params = filter(lambda p: p.requires_grad, model.parameters())
optimizer = AdamW(trainable_params, lr=5e-5)

# INEFFICIENT training process example (as shown in notebook)
# Problems highlighted:
# 1. Iterates sample by sample (no batching via DataLoader).
# 2. Tokenizes *inside* the loop (redundant computation).
# 3. No explicit device management (runs on CPU unless manually moved).
# 4. Fixed number of epochs might be too much without validation.
# 5. Uses the `eval_dataset` for iteration demonstration - normally you'd iterate `train_dataset`.
print("\n--- Starting Inefficient Loop Demo (will be slow) ---")
num_epochs_inefficient = 1 # Reduced drastically for demo speed
inefficient_counter = 0
max_inefficient_steps = 10 # Limit steps for demo

model.to("cpu") # Ensure model is on CPU for this specific inefficient example if GPU was used before

for epoch in range(num_epochs_inefficient):
    print(f"Inefficient Epoch {epoch+1}/{num_epochs_inefficient}")
    # Using eval_dataset just to have a small iterable; normally use train_dataset
    for batch_item in eval_dataset: # INEFFICIENT: Iterating single items
        if inefficient_counter >= max_inefficient_steps:
             break

        # INEFFICIENT: Repeated tokenization inside loop for each example
        # Ensure padding/truncation match the efficient approach for fair comparison if running longer
        inputs = tokenizer(batch_item["text"], return_tensors="pt", padding=True, truncation=True, max_length=128)
        inputs['labels'] = torch.tensor([batch_item["label"]]) # Create label tensor

        # INEFFICIENT: No explicit device placement (will run on CPU here)
        outputs = model(**inputs)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad() # Correct placement for single item update

        inefficient_counter += 1
        if inefficient_counter % 5 == 0: # Log progress infrequently
            print(f"  Inefficient step {inefficient_counter}, Loss: {loss.item()}")

    if inefficient_counter >= max_inefficient_steps:
        print(f"Reached max {max_inefficient_steps} inefficient steps for demo.")
        break
print("--- Finished Inefficient Loop Demo ---")




--- Starting Inefficient Loop Demo (will be slow) ---
Inefficient Epoch 1/1
  Inefficient step 5, Loss: 1.199262261390686
  Inefficient step 10, Loss: 0.446157306432724
Reached max 10 inefficient steps for demo.
--- Finished Inefficient Loop Demo ---


In [8]:
from torch.utils.data import DataLoader

# Create DataLoader for efficient batching and shuffling
# Use the actual small train_dataset created in Step 4
train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True)

# Fresh optimizer for this specific loop
# Ensure we capture parameters that are currently trainable
trainable_params = filter(lambda p: p.requires_grad, model.parameters())
optimizer = AdamW(trainable_params, lr=5e-5) # Reinitialize optimizer state

# Determine device (CPU or GPU) and move model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("\n--- Starting Efficient Loop Demo ---")
num_epochs_efficient = 1 # Reduced epochs appropriate for small dataset/demo

for epoch in range(num_epochs_efficient):
    print(f"Efficient Epoch {epoch+1}/{num_epochs_efficient}")
    model.train() # Set model to training mode
    step_counter = 0
    for batch in train_dataloader: # EFFICIENT: Iterating over batches from DataLoader
        optimizer.zero_grad() # Zero gradients *before* the batch processing

        # Prepare inputs (assuming batch contains 'text' and 'label' from original dataset structure)
        # This assumes the DataLoader is yielding dictionaries with these keys
        # In a real scenario with pre-tokenized data, keys would be 'input_ids', 'attention_mask', etc.
        inputs = tokenizer(batch["text"], return_tensors="pt", padding=True, truncation=True, max_length=128) # Tokenize the batch text
        labels = batch["label"] # Get labels from the batch
        inputs['labels'] = labels # Add labels to the inputs dictionary

        # EFFICIENT: Move batch data to the same device as the model
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Forward pass
        outputs = model(**inputs)
        loss = outputs.loss

        # Backward pass
        loss.backward()

        # Optimizer step
        optimizer.step()

        step_counter += 1
        if step_counter % 10 == 0: # Log every 10 steps
             print(f"  Efficient step {step_counter}, Loss: {loss.item()}") # Log loss

print("--- Finished Efficient Loop Demo ---")



--- Starting Efficient Loop Demo ---
Efficient Epoch 1/1
  Efficient step 10, Loss: 0.5725871920585632
  Efficient step 20, Loss: 0.5644770860671997
  Efficient step 30, Loss: 0.24911636114120483
  Efficient step 40, Loss: 0.3695172071456909
  Efficient step 50, Loss: 0.25939589738845825
  Efficient step 60, Loss: 0.09855134785175323
--- Finished Efficient Loop Demo ---


In [11]:
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import requests # To fetch image from URL
from io import BytesIO # To handle image bytes

# Load pretrained CLIP model and processor
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# URL of an image (replace with a working URL or use the example)
# Example: A picture of cats playing
#image_url = "https://unionlakeveterinaryhospital.com/wp-content/uploads/2021/04/ULVH-cats-playing-shutterstock_1211910061.jpeg"
# Example: A picture of a dog
image_url = "https://images.pexels.com/photos/1108099/pexels-photo-1108099.jpeg"

print(f"\n--- Starting CLIP Demo ---")
try:
    # Fetch the image from the URL
    print(f"Fetching image from: {image_url}")
    response = requests.get(image_url)
    response.raise_for_status() # Check if the request was successful
    image = Image.open(BytesIO(response.content))
    print("Image fetched successfully.")

    # Process the image and text prompts
    # Using slightly different prompts than the notebook for clarity
    text_prompts = ["a photo of cats playing", "a photo of a dog running"]
    print(f"Processing image with text prompts: {text_prompts}")
    inputs = processor(text=text_prompts, images=image, return_tensors="pt", padding=True)

    # Get similarity scores from the model
    # Move model to appropriate device if necessary (CLIP models are large)
    clip_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    clip_model.to(clip_device)
    inputs = {k: v.to(clip_device) for k, v in inputs.items()}

    with torch.no_grad(): # Disable gradient calculation for inference
        outputs = clip_model(**inputs)

    logits_per_image = outputs.logits_per_image # Logits representing similarity
    probs = logits_per_image.softmax(dim=1) # Convert logits to probabilities

    print(f"\nResults for image: {image_url}")
    print(f"Similarity Logits (Image vs {text_prompts}): {logits_per_image.cpu().numpy()}")
    print(f"Similarity Probabilities (Image vs {text_prompts}): {probs.cpu().numpy()}")

except Exception as e:
    print(f"Error processing image from URL: {e}")
    print("Please ensure the image_url is valid and accessible.")
print("--- Finished CLIP Demo ---")



--- Starting CLIP Demo ---
Fetching image from: https://images.pexels.com/photos/1108099/pexels-photo-1108099.jpeg
Image fetched successfully.
Processing image with text prompts: ['a photo of cats playing', 'a photo of a dog running']

Results for image: https://images.pexels.com/photos/1108099/pexels-photo-1108099.jpeg
Similarity Logits (Image vs ['a photo of cats playing', 'a photo of a dog running']): [[20.42841  25.791119]]
Similarity Probabilities (Image vs ['a photo of cats playing', 'a photo of a dog running']): [[0.00466631 0.99533373]]
--- Finished CLIP Demo ---


In [12]:
!pip install faiss-cpu transformers datasets torch


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 69.9 MB/s eta 0:00:00:00:0100:01

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [13]:
from transformers import AutoTokenizer, AutoModel
import torch
import faiss
import numpy as np

# Load embedding model
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

# Example documents
documents = [
    "The Eiffel Tower is located in Paris, France.",
    "The capital of Germany is Berlin.",
    "The Great Wall of China is a famous historical landmark."
]

# Convert documents to embeddings
def embed_text(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        embeddings = model(**inputs).last_hidden_state.mean(dim=1)
    return embeddings.squeeze().numpy()

# Create document embeddings
embeddings = np.array([embed_text(doc) for doc in documents])


In [14]:
# Initialize FAISS index
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)


In [15]:
query = "Where is the Eiffel Tower?"
query_embedding = embed_text(query).reshape(1, -1)
D, I = index.search(query_embedding, k=1)  # Retrieve top-1 match

print("Retrieved Document:", documents[I[0][0]])


Retrieved Document: The Eiffel Tower is located in Paris, France.


In [16]:
query = "What is RAG"

query_embedding = embed_text(query).reshape(1, -1)

distance_threshold = 10

D, I = index.search(query_embedding, k=1)

if len(I[0]) > 0 and D[0][0] < distance_threshold:
    retrieved_index = I[0][0]
    retrieved_doc = documents[retrieved_index]
    print("Retrieved Document:", retrieved_doc)
else:
    print("I can't answer that.")


I can't answer that.
